# Modelado (Fase 3)

Este notebook reúne la Fase 3: los baselines obligatorios, la comparación de al menos cuatro modelos, la elección del modelo final y su registro. Se arma por secciones, y cada integrante agrega las suyas en una rama propia. La sección 1, que es la de este primer paso, fija las reglas con las que se miden todos los modelos: sobre qué filas se evalúa, qué errores tienen los dos baselines y cómo se registra cada experimento. Nada de lo que se calcula acá toca el tramo de prueba, que se mira una sola vez, al final, con el modelo ya elegido.

## 1. Datos, baselines y reglas de evaluación

Los modelos parten del dataset final de la Fase 2 (`data/processed/dataset_final.csv`), que tiene las 14 variables elegidas en sus unidades originales, el objetivo (el consumo de frío del día siguiente) y el tramo de cada fila. Se lo carga con `cargar_dataset_final`, que por defecto no incluye el tramo de prueba: así nadie lo usa por descuido para elegir un modelo. La tabla 1.1 muestra cuántas filas y qué fechas tiene cada tramo de trabajo.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import huella, tabla_simple
from modelado_comun import (RUTA_DATASET_FINAL, RUTA_LOG, cargar_dataset_final, evaluar_baselines, filas_comunes,
                            registrar_baselines, validacion_cruzada_temporal)

RAIZ = Path("..").resolve()
datos = cargar_dataset_final()                       # solo entrenamiento y validación: la prueba no se carga

def titulo_y_tabla(numero, titulo, tabla, precision=2):
    """Imprime el título de la tabla y la muestra con el estilo del proyecto."""
    print(f"Tabla 1.{numero}. {titulo}")
    display(tabla_simple(tabla, precision=precision))

tramos = pd.DataFrame({
    tramo: {"Filas": len(d["y"]), "Variables": d["X"].shape[1],
            "Primera fecha": d["fechas"].min().date().isoformat(), "Última fecha": d["fechas"].max().date().isoformat()}
    for tramo, d in datos.items()}).T
tramos.index.name = "Tramo"
titulo_y_tabla(1, "Tramos de trabajo del dataset final (la prueba no se carga)", tramos, precision=0)

Tabla 1.1. Tramos de trabajo del dataset final (la prueba no se carga)


,Filas,Variables,Primera fecha,Última fecha
Tramo,,,,
entrenamiento,830,14,2020-07-01,2022-10-27
validacion,176,14,2022-10-29,2023-04-26


Los dos baselines son predicciones sin ningún parámetro: la persistencia dice que mañana se consume lo mismo que hoy, y la media móvil, que mañana se consume el promedio de los últimos siete días (con al menos cuatro días válidos). Como no se ajustan con nada, no hay riesgo de que aprendan del tramo donde se miden. Se miden de todos modos en entrenamiento y en validación porque son la vara contra la que se compara cada modelo: un modelo que se evalúa en un conjunto de filas tiene que compararse con los baselines en esas mismas filas.

Por eso se define un conjunto de filas comunes, que son las que tienen objetivo y las dos predicciones de los baselines. Quedan afuera los días inválidos y los primeros días de la serie, donde la media móvil todavía no tiene suficientes días. Todos los modelos se evalúan en esas filas, y la tabla 1.2 cuenta cuántas quedan y cuántas se excluyen en cada tramo.

In [2]:
comunes = pd.DataFrame({
    tramo: {"Filas del tramo": len(d["y"]), "Filas comunes": int(filas_comunes(d).sum()),
            "Sin persistencia (hoy inválido)": int(d["persistencia"].isna().sum()),
            "Sin media móvil (pocos días válidos)": int(d["media_movil"].isna().sum())}
    for tramo, d in datos.items()}).T
comunes.index.name = "Tramo"
titulo_y_tabla(2, "Filas sobre las que se compara todo (las que tienen objetivo y los dos baselines)", comunes, precision=0)

Tabla 1.2. Filas sobre las que se compara todo (las que tienen objetivo y los dos baselines)


,Filas del tramo,Filas comunes,Sin persistencia (hoy inválido),Sin media móvil (pocos días válidos)
Tramo,,,,
entrenamiento,830,813,14,3
validacion,176,171,5,0


La tabla 1.3 da el error absoluto medio de cada baseline en esas filas. En entrenamiento es el promedio de todo el tramo, y en validación es un único bloque de fechas, con otra época del año, otro nivel de consumo y otra variabilidad, así que los dos números no se pueden comparar entre sí: lo que importa es comparar un modelo con los baselines dentro del mismo tramo. Estos valores se registran como los primeros experimentos de `results/experiment_logs.csv`, y volver a ejecutar la celda no repite las filas.

In [3]:
resultados = pd.DataFrame({tramo: evaluar_baselines(d) for tramo, d in datos.items()}).T
resultados.index.name = "Tramo"
resultados = resultados.rename(columns={"filas": "Filas comunes", "persistencia": "MAE persistencia (kW)",
                                        "media_movil": "MAE media móvil (kW)"})
titulo_y_tabla(3, "Error absoluto medio de los baselines sobre las filas comunes", resultados)

nuevas = registrar_baselines(datos)                  # idempotente: si ya están en el log, no las repite
print(f"Filas agregadas hoy al registro de experimentos: {len(nuevas)}")

Tabla 1.3. Error absoluto medio de los baselines sobre las filas comunes


,Filas comunes,MAE persistencia (kW),MAE media móvil (kW)
Tramo,,,
entrenamiento,813.00,3185.88,3293.61
validacion,171.00,2668.33,2497.65


Filas agregadas hoy al registro de experimentos: 4


Para ajustar hiperparámetros dentro del entrenamiento, todos los modelos usan el mismo divisor de pliegues cronológicos, `validacion_cruzada_temporal`, que no mezcla el orden y deja un día de separación entre lo que ajusta y lo que evalúa. La tabla 1.4 muestra, para cada pliegue, qué fechas evalúa y el error de los baselines en sus filas comunes. Sirve de referencia: un modelo se mide pliegue por pliegue contra estos números, y el error cambia de un pliegue a otro porque cada uno cubre otra época y entrena con más historia que el anterior.

In [4]:
entrenamiento = datos["entrenamiento"]
filas_pliegues = []
for numero, (ajuste, evaluacion) in enumerate(validacion_cruzada_temporal().split(entrenamiento["X"]), 1):
    parte = {clave: (valor.iloc[evaluacion] if hasattr(valor, "iloc") else valor) for clave, valor in entrenamiento.items()}
    base = evaluar_baselines(parte)
    filas_pliegues.append({"Pliegue": numero, "Filas de ajuste": len(ajuste), "Filas de evaluación": len(evaluacion),
                           "Filas comunes": base["filas"],
                           "Inicio": parte["fechas"].min().date().isoformat(), "Fin": parte["fechas"].max().date().isoformat(),
                           "MAE persistencia (kW)": base["persistencia"], "MAE media móvil (kW)": base["media_movil"]})
pliegues = pd.DataFrame(filas_pliegues).set_index("Pliegue")
titulo_y_tabla(4, "Los cinco pliegues cronológicos del entrenamiento y el error de los baselines en cada uno", pliegues)

Tabla 1.4. Los cinco pliegues cronológicos del entrenamiento y el error de los baselines en cada uno


,Filas de ajuste,Filas de evaluación,Filas comunes,Inicio,Fin,MAE persistencia (kW),MAE media móvil (kW)
Pliegue,,,,,,,
1,139,138,138,2020-11-22,2021-04-06,2576.87,2858.93
2,277,138,131,2021-04-07,2021-09-01,3478.95,3301.24
3,415,138,137,2021-09-02,2022-01-18,3240.04,3774.99
4,553,138,134,2022-01-19,2022-06-09,2976.93,3143.01
5,691,138,137,2022-06-10,2022-10-27,3974.88,3805.95


La tabla 1.5 reúne los controles de esta sección y todos tienen que dar verdadero: que la prueba no se cargó, que la huella del dataset coincide con la guardada en `data/checksums.json`, que las filas comunes de validación son las mismas que se midieron en la Fase 2 para comparar los pipelines, que cada pliegue evalúa fechas posteriores a las que ajusta y que los baselines quedaron registrados.

In [5]:
import json

huellas = json.loads((RAIZ / "data" / "checksums.json").read_text(encoding="utf-8"))
registro = pd.read_csv(RUTA_LOG)
baselines_registrados = registro.loc[registro["notas"] == "baseline sobre las filas comunes", ["modelo", "tramo"]]
controles = pd.Series({
    "La prueba no se cargó": "prueba" not in datos,
    "La huella del dataset final coincide con data/checksums.json": huella(RUTA_DATASET_FINAL) == huellas["processed/dataset_final.csv"],
    "Las filas comunes de validación son las 171 de la Fase 2": int(filas_comunes(datos["validacion"]).sum()) == 171,
    "Cada pliegue evalúa fechas posteriores a las que ajusta": all(
        entrenamiento["fechas"].iloc[ajuste].max() <= entrenamiento["fechas"].iloc[evaluacion].min()
        for ajuste, evaluacion in validacion_cruzada_temporal().split(entrenamiento["X"])),
    "Los cuatro baselines (2 por tramo) están en el registro": len(baselines_registrados.drop_duplicates()) == 4})
titulo_y_tabla(5, "Controles de la sección (todos deben ser True)", controles.to_frame("Cumple"), precision=0)
assert controles.all(), controles[~controles]

Tabla 1.5. Controles de la sección (todos deben ser True)


,Cumple
La prueba no se cargó,True
La huella del dataset final coincide con data/checksums.json,True
Las filas comunes de validación son las 171 de la Fase 2,True
Cada pliegue evalúa fechas posteriores a las que ajusta,True
Los cuatro baselines (2 por tramo) están en el registro,True


### Conclusión de la sección

Quedaron fijadas las reglas de la Fase 3. Todos los modelos usan las mismas 14 variables, se evalúan sobre las mismas filas comunes que los baselines, se ajustan dentro del entrenamiento con los mismos pliegues cronológicos y se comparan por su error en validación; el tramo de prueba no se carga ni se registra hasta la evaluación final. Los dos baselines ya están medidos y registrados, y son lo que cualquier modelo tiene que superar en las mismas filas.

Hay que tener presente que los baselines de entrenamiento y de validación no se pueden comparar entre sí, y que la validación ya se usó antes para elegir el método de selección de variables, de modo que el error de un modelo en validación es algo optimista. Lo que sigue es agregar, en ramas propias, la sección de cada modelo (XGBoost, LightGBM, Random Forest y Ridge o Lasso) usando estas mismas funciones.